# Briscola Benchmark Analysis
Runs the Rust benchmark binary and loads results directly into pandas — no temp files.

In [ ]:
import io
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

# notebooks/ -> briscola/ -> cardroom-python/ -> repo root
REPO_ROOT = Path.cwd().resolve().parents[2]
WORKSPACE = REPO_ROOT / "cardroom"  # the Rust cargo workspace
BINARY = WORKSPACE / "target" / "release" / "examples" / "benchmark"

sns.set_theme(style="darkgrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
def build():
    """Build the release binary. Only needed once per code change."""
    subprocess.run(
        ["cargo", "build", "--release", "--example", "benchmark"],
        cwd=WORKSPACE, check=True,
    )
    print("Build OK:", BINARY)


def run_benchmark(agent0: str, agent1: str, n_rounds: int = 1000) -> pd.DataFrame:
    """Run the benchmark and return per-trick data as a DataFrame.

    Progress is printed to stderr (visible in the notebook).
    CSV data is captured from stdout.
    """
    result = subprocess.run(
        [str(BINARY), agent0, agent1, str(n_rounds), "--csv"],
        capture_output=True, text=True, cwd=WORKSPACE, check=True,
    )
    df = pd.read_csv(io.StringIO(result.stdout))
    df["had_briscola"] = df["had_briscola"].astype(bool)
    df["agent0_took_trick"] = df["agent0_took_trick"].astype(bool)
    return enrich(df)


# Points by card face (Ace/Three are the "carichi"). Unlisted faces score 0.
FACE_POINTS = {"A": 11, "3": 10, "10": 4, "9": 3, "8": 2}


def enrich(df: pd.DataFrame) -> pd.DataFrame:
    """Derive per-trick, per-agent features from the card + order columns."""
    df = df.copy()
    for a in (0, 1):
        card = df[f"agent{a}_card"].str.split(" of ", expand=True)
        df[f"agent{a}_face"] = card[0]
        df[f"agent{a}_suit"] = card[1]
        df[f"agent{a}_points"] = df[f"agent{a}_face"].map(FACE_POINTS).fillna(0).astype(int)
        df[f"agent{a}_is_briscola"] = df[f"agent{a}_suit"] == df["briscola_suit"]
        # carico = Ace or Three NOT of the briscola suit
        df[f"agent{a}_is_carico"] = (
            df[f"agent{a}_face"].isin(["A", "3"]) & ~df[f"agent{a}_is_briscola"]
        )
    # Every trick has exactly one winner (2 players)
    df["agent1_took_trick"] = ~df["agent0_took_trick"]
    df["agent0_first"] = df["first_agent"] == 0
    df["agent1_first"] = df["first_agent"] == 1
    # Signed points netted by each agent in the trick:
    #   +trick_points if the agent took it, -trick_points if the opponent did.
    for a in (0, 1):
        took = df[f"agent{a}_took_trick"]
        df[f"agent{a}_signed"] = df["trick_points"].where(took, -df["trick_points"])
    return df


In [ ]:
# Build once; comment out after first run if you're only changing the notebook
build()

In [ ]:
# --- Configure your matchup here ---
AGENT0 = "bot"
AGENT1 = "mcts"
N_ROUNDS = 2000

df = run_benchmark(AGENT0, AGENT1, N_ROUNDS)
print(f"{len(df):,} trick rows  ({N_ROUNDS * 2:,} games)")
df.head()

## Win rates & average final points

In [ ]:
# One row per game (use last trick of each game)
games = df.groupby(["round", "game"]).last().reset_index()

n_games = len(games)
wins0 = (games["game_winner"] == "agent0").sum()
wins1 = (games["game_winner"] == "agent1").sum()
draws = (games["game_winner"] == "draw").sum()

print(f"Games: {n_games}")
print(f"{AGENT0:>10}  wins: {wins0:5}  ({wins0/n_games*100:.1f}%)")
print(f"{AGENT1:>10}  wins: {wins1:5}  ({wins1/n_games*100:.1f}%)")
print(f"{'Draw':>10}       : {draws:5}  ({draws/n_games*100:.1f}%)")
print()
print(f"Avg final pts  {AGENT0}: {games['agent0_final'].mean():.1f}")
print(f"Avg final pts  {AGENT1}: {games['agent1_final'].mean():.1f}")

# Points distribution
fig, ax = plt.subplots(figsize=(10, 5))
bins = range(0, 125, 5)
ax.hist(games["agent0_final"], bins=bins, alpha=0.6, label=AGENT0)
ax.hist(games["agent1_final"], bins=bins, alpha=0.6, label=AGENT1)
ax.axvline(60, color="gray", linestyle="--", linewidth=1, label="60 pts")
ax.set_xlabel("Final points")
ax.set_ylabel("Games")
ax.set_title(f"Final score distribution — {AGENT0} vs {AGENT1} ({n_games:,} games)")
ax.legend()
plt.tight_layout()
plt.show()

## Points progression across tricks

In [ ]:
progression = df.groupby("trick")[["agent0_cumpts", "agent1_cumpts"]].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(progression.index, progression["agent0_cumpts"], label=AGENT0, linewidth=2)
ax.plot(progression.index, progression["agent1_cumpts"], label=AGENT1, linewidth=2)
ax.axhline(60, color="gray", linestyle="--", linewidth=1, label="Draw line (60 pts)")
ax.set_xlabel("Trick number")
ax.set_ylabel("Avg cumulative points")
ax.set_title(f"Points progression — {AGENT0} vs {AGENT1} ({n_games:,} games)")
ax.legend()
ax.set_xticks(range(1, 21))
plt.tight_layout()
plt.show()

## Match result — statistical significance

Two-sided binomial test on decisive games (draws excluded): is the win-count difference between the two players real, or just noise?

In [ ]:
# One row per game already computed above as `games`.
n_games = len(games)
wins0 = int((games["game_winner"] == "agent0").sum())
wins1 = int((games["game_winner"] == "agent1").sum())
draws = int((games["game_winner"] == "draw").sum())
decisive = wins0 + wins1

# Two-sided binomial test on decisive games: is either player favored over 50%?
res = stats.binomtest(wins0, decisive, 0.5, alternative="two-sided")
ci = res.proportion_ci(confidence_level=0.95)
alpha = 0.05

print(f"Games: {n_games}   decisive: {decisive}   draws: {draws}")
print(f"{AGENT0} wins: {wins0}    {AGENT1} wins: {wins1}")
print(f"{AGENT0} win share (decisive): {wins0/decisive:.3f}  "
      f"95% CI [{ci.low:.3f}, {ci.high:.3f}]")
print(f"Binomial test (H0: 50/50) p-value: {res.pvalue:.3g}")
if res.pvalue < alpha:
    leader = AGENT0 if wins0 > wins1 else AGENT1
    print(f"=> SIGNIFICANT at alpha={alpha}: {leader} wins more often than chance.")
else:
    print(f"=> NOT significant at alpha={alpha}: no detectable win-rate difference.")

# Win counts with a 95% CI band (CI is on agent0's share, mirrored for agent1)
counts = [wins0, wins1]
ci_low = [decisive * ci.low, decisive * (1 - ci.high)]
ci_high = [decisive * ci.high, decisive * (1 - ci.low)]
yerr = [[c - lo for c, lo in zip(counts, ci_low)],
        [hi - c for c, hi in zip(counts, ci_high)]]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar([AGENT0, AGENT1], counts, color=["C0", "C1"], alpha=0.85,
              yerr=yerr, capsize=8)
ax.axhline(decisive / 2, color="gray", linestyle="--", linewidth=1,
           label="50% of decisive games")
ax.bar_label(bars, padding=3)
verdict = "significant" if res.pvalue < alpha else "not significant"
ax.set_ylabel("Wins (decisive games)")
ax.set_title(f"Match wins — {AGENT0} vs {AGENT1}\n"
             f"binomial p={res.pvalue:.3g} ({verdict}); {draws} draws excluded")
ax.legend()
plt.tight_layout()
plt.show()


## Points gained per trick

In [ ]:
gained = df.copy()
# Compute per-trick gain from cumulative columns
gained = gained.sort_values(["round", "game", "trick"])
gained["agent0_gained"] = gained.groupby(["round", "game"])["agent0_cumpts"].diff().fillna(gained["agent0_cumpts"])
gained["agent1_gained"] = gained.groupby(["round", "game"])["agent1_cumpts"].diff().fillna(gained["agent1_cumpts"])

avg_gained = gained.groupby("trick")[["agent0_gained", "agent1_gained"]].mean()

fig, ax = plt.subplots(figsize=(10, 5))
x = avg_gained.index
width = 0.4
ax.bar(x - width/2, avg_gained["agent0_gained"], width, label=AGENT0)
ax.bar(x + width/2, avg_gained["agent1_gained"], width, label=AGENT1)
ax.set_xlabel("Trick number")
ax.set_ylabel("Avg points gained")
ax.set_title(f"Avg points gained per trick — {AGENT0} vs {AGENT1}")
ax.legend()
ax.set_xticks(range(1, 21))
plt.tight_layout()
plt.show()

## Briscola and carico EV

Expected **signed** points netted on tricks where a player plays a briscola/carico card (`+trick_points` if they take the trick, `-trick_points` if the opponent does).

In [ ]:
def signed_value(df, mask_tmpl):
    """Mean signed trick points per agent over tricks matching a per-agent mask."""
    rows = []
    for a, name in [(0, AGENT0), (1, AGENT1)]:
        mask = df[mask_tmpl.format(a)]
        rows.append({
            "agent": name,
            "ev": df.loc[mask, f"agent{a}_signed"].mean(),
            "n": int(mask.sum()),
        })
    return pd.DataFrame(rows)


def plot_value(vdf, title, ax):
    """Draw a signed-value bar chart onto the given axis."""
    bars = ax.bar(vdf["agent"], vdf["ev"], color=["C0", "C1"], alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.bar_label(bars, fmt="%.2f", padding=3)
    for i, n in enumerate(vdf["n"]):
        ax.annotate(f"n={n}", (i, 0), textcoords="offset points", xytext=(0, -16),
                    ha="center", fontsize=9, color="dimgray")
    ax.set_ylabel("Avg signed points per trick")
    ax.set_title(title)


bris = signed_value(df, "agent{}_is_briscola")
print("BRISCOLA")
print(bris.to_string(index=False))

caro = signed_value(df, "agent{}_is_carico")
print("CARICHI")
print(caro.to_string(index=False))


## First-of-turn EV

Expected signed points per trick when a player plays **first** vs **second** in the trick.

In [ ]:
rows = []
for a, name in [(0, AGENT0), (1, AGENT1)]:
    for is_first, pos in [(True, "Play first"), (False, "Play second")]:
        mask = df[f"agent{a}_first"] == is_first
        rows.append({
            "agent": name,
            "position": pos,
            "ev": df.loc[mask, f"agent{a}_signed"].mean(),
            "n": int(mask.sum()),
        })
pos_df = pd.DataFrame(rows)
print(pos_df.to_string(index=False))

# --- Combined figure: briscola value, carico value, first-of-turn EV ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_value(bris, f"Briscola value\nEV when playing a briscola", axes[0])
plot_value(caro, f"Carico value\nEV when playing a carico (A/3, non-briscola)", axes[1])

# First-of-turn EV (grouped bars) on the third axis
ax = axes[2]
positions = ["Play first", "Play second"]
x = range(len(positions))
width = 0.4
for j, name in enumerate([AGENT0, AGENT1]):
    vals = [pos_df[(pos_df.agent == name) & (pos_df.position == p)]["ev"].iloc[0]
            for p in positions]
    offset = (j - 0.5) * width
    b = ax.bar([xi + offset for xi in x], vals, width, label=name, color=f"C{j}", alpha=0.85)
    ax.bar_label(b, fmt="%.2f", padding=3)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(positions)
ax.set_ylabel("Avg signed points per trick")
ax.set_title("First-of-turn EV\npoints by play position")
ax.legend()

fig.suptitle(f"{AGENT0} vs {AGENT1}", fontsize=14)
plt.tight_layout()
plt.show()


## Mirror-pair concordance

Each **round** is the *same deal played twice from opposite sides*. In game 2 the agents swap cards: each one plays the hand (and the full draw sequence) its opponent held in game 1 — only the briscola-spy draw is left uncontrolled. This way card luck is comparable within a pair.

We compare the two outcomes of each pair:

- **Concordant** — the result *flipped*: one agent won game 1, the other won game 2. The same side of the deal won both times, just in different hands → **card luck decided the round**, not skill.
- **Discordant** — the result did *not* flip; the same agent prevailed on *both* sides of the deal → **skill overcame the cards**:
  - **Hard discordant** — same agent won **both** games (`win/win` or `loss/loss`).
  - **Soft discordant** — one decisive game + one draw (`win/draw` or `loss/draw`).
  - *(both games drawn is reported separately.)*

A clean, luck-free skill signal is the **discordant rate** (and the per-agent hard/soft split): the more often one agent wins regardless of which side of the deal it holds, the stronger it is.


In [ ]:
# One outcome per game -> one row per round with both game outcomes side by side.
pairs = games.pivot(index="round", columns="game", values="game_winner")
pairs.columns = ["g1", "g2"]
assert pairs.notna().all().all(), "every round must have both mirrored games"


def classify(g1, g2):
    s = {g1, g2}
    if s == {"agent0", "agent1"}:          # one agent won each game
        return "concordant"                # -> result flipped
    if g1 == g2 == "agent0":
        return "hard_a0"                   # agent0 won both
    if g1 == g2 == "agent1":
        return "hard_a1"                   # agent1 won both
    if g1 == g2 == "draw":
        return "both_draw"
    decisive = g1 if g1 != "draw" else g2  # one decisive + one draw
    return "soft_a0" if decisive == "agent0" else "soft_a1"


pairs["klass"] = [classify(g1, g2) for g1, g2 in zip(pairs["g1"], pairs["g2"])]

n_pairs = len(pairs)
c = lambda k: int((pairs["klass"] == k).sum())
concordant = c("concordant")
hard_a0, hard_a1 = c("hard_a0"), c("hard_a1")
soft_a0, soft_a1 = c("soft_a0"), c("soft_a1")
both_draw = c("both_draw")
hard, soft = hard_a0 + hard_a1, soft_a0 + soft_a1
discordant = hard + soft + both_draw


def line(label, n):
    print(f"  {label:<24} {n:5}  ({n / n_pairs * 100:5.1f}%)")


print(f"Mirror pairs (rounds): {n_pairs:,}\n")
line("Concordant (flipped)", concordant)
line("Discordant (not flipped)", discordant)
print("\nHard discordant  — same agent won BOTH games:")
line(f"{AGENT0} win / win", hard_a0)
line(f"{AGENT1} win / win", hard_a1)
line("total", hard)
print("\nSoft discordant  — one win + one draw:")
line(f"{AGENT0} win / draw", soft_a0)
line(f"{AGENT1} win / draw", soft_a1)
line("total", soft)
print("\nOther:")
line("both games drawn", both_draw)

# Full game1 x game2 outcome matrix (diagonal = no flip, anti-diagonal = flip)
label = {"agent0": AGENT0, "agent1": AGENT1, "draw": "draw"}
order = [AGENT0, AGENT1, "draw"]
ct = (
    pd.crosstab(pairs["g1"].map(label), pairs["g2"].map(label))
    .reindex(index=order, columns=order, fill_value=0)
)
ct.index.name, ct.columns.name = "game 1 winner", "game 2 winner"
print("\nGame 1 x Game 2 outcome counts:")
print(ct)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: heatmap of the game1 x game2 outcome matrix.
# Anti-diagonal corners (agentX, agentY) = flips; the rest = no flip.
sns.heatmap(ct, annot=True, fmt=",d", cmap="Blues", cbar=False,
            linewidths=0.5, linecolor="white", ax=axes[0])
axes[0].set_title(f"Game 1 x Game 2 outcomes ({n_pairs:,} pairs)")

# Right: category breakdown, readable per agent.
cats = [
    ("Concordant\n(flipped)",      concordant, "C2"),
    (f"Hard — {AGENT0}\nwin/win",  hard_a0,    "C0"),
    (f"Hard — {AGENT1}\nwin/win",  hard_a1,    "C1"),
    (f"Soft — {AGENT0}\nwin/draw", soft_a0,    "C0"),
    (f"Soft — {AGENT1}\nwin/draw", soft_a1,    "C1"),
    ("Both\ndrawn",                both_draw,  "C7"),
]
labels = [c[0] for c in cats]
vals = [c[1] for c in cats]
colors = [c[2] for c in cats]
# Hatch the soft bars to distinguish them from the hard bars of the same agent.
hatches = [None, None, None, "//", "//", None]

bars = axes[1].bar(labels, vals, color=colors, alpha=0.85)
for b, h in zip(bars, hatches):
    if h:
        b.set_hatch(h)
axes[1].bar_label(bars, labels=[f"{v}\n{v / n_pairs * 100:.1f}%" for v in vals], padding=3)
axes[1].set_ylabel("Pairs")
axes[1].set_title("Pair concordance breakdown")
axes[1].margins(y=0.15)

fig.suptitle(f"Mirror-pair concordance — {AGENT0} vs {AGENT1}", fontsize=14)
plt.tight_layout()
plt.show()
